# PepXPro ML Benchmark

**Question:** Does a PepXPro-curated benchmark (Kennard–Stone selected) reveal more about model
generalization than a random sample of the same size?

**Design:**
- Split the full dataset into 80% training / 20% held-out pool (repeated 25 seeds)
- From the held-out pool, select two test sets of equal size (n=80):
  - **Test A — Random:** pure random sample
  - **Test B — PepXPro:** peptide sequence dedup + Kennard–Stone on the 11 structural features
- Train the **same** Random Forest on the fixed 80% training set
- Compare RMSE, internal diversity, and dG coverage between test sets A and B

**Expected outcome:**
- PepXPro test set is harder (higher RMSE) because KS selects entries at the edges of feature space
- PepXPro test set has higher internal diversity (mean pairwise distance in feature space)
- PepXPro test set covers a wider range of ΔG values

**Structural feature:** `n_contacts` — peptide heavy atoms within 4.5 Å of any receptor heavy atom (from `fixed_cif/`).

## 1. Setup

In [ ]:
import sys, time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from difflib import SequenceMatcher
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Paths ───────────────────────────────────────────────────────────────────
REPO         = Path("../..").resolve()
SCRAPE       = REPO / "scrape_pdb_bind"
SUMMARY_TSV  = SCRAPE / "outputs" / "summary.tsv"
FILTERED_TSV = SCRAPE / "outputs" / "filtered_systems.tsv"
FIXED_DIR    = SCRAPE / "fixed_cif"
OUT_DIR      = Path("..") / "outputs"
PLOT_DIR     = Path("..") / "plots"
OUT_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)
CONTACTS_CACHE = OUT_DIR / "contacts_cache.tsv"

# ── Configuration ───────────────────────────────────────────────────────────
N_TEST           = 80      # size of each test set
N_SEEDS          = 25      # number of train/test split repetitions
CONTACT_CUTOFF_A = 4.5     # Å
PEP_DEDUP_THRESH = 0.80
TRAIN_FRAC       = 0.80

FEATURES = [
    "pep_len", "rec_len",
    "peptide_pct_helix", "peptide_pct_beta", "peptide_pct_coil",
    "receptor_pct_helix", "receptor_pct_beta", "receptor_pct_coil",
    "n_contacts", "binding_kd", "resolution_f",
]
TARGET = "dG_kcal_mol"

COLOR_A = "#4c72b0"   # random  → blue
COLOR_B = "#c44e52"   # pepxpro → red

print(f"Repository : {REPO}")
print(f"N_TEST     : {N_TEST}")
print(f"N_SEEDS    : {N_SEEDS}")
print(f"Features   : {len(FEATURES)}")

## 2. Load data

In [ ]:
summary  = pd.read_csv(SUMMARY_TSV, sep="\t")
filtered = pd.read_csv(FILTERED_TSV, sep="\t")
filtered_acc = filtered[filtered["accepted"] == "Y"].set_index("pdb_id")

summary["pep_len"]      = summary["peptide"].str.split(":").str[1].astype(float)
summary["rec_len"]      = summary["receptor"].str.split(":").str[1].astype(float)
summary["resolution_f"] = pd.to_numeric(summary["resolution"], errors="coerce")
summary["binding_kd"]   = (summary["binding_type"] == "Kd").astype(float)
for col in ["peptide_pct_helix","peptide_pct_beta","peptide_pct_coil",
            "receptor_pct_helix","receptor_pct_beta","receptor_pct_coil"]:
    summary[col] = pd.to_numeric(summary[col], errors="coerce").fillna(0.0)

print(f"Total entries: {len(summary)}")
print(f"\nreceptor_func_class distribution:")
print(summary["receptor_func_class"].value_counts().to_string())

## 3. Compute contacts

In [ ]:
def compute_contacts(pdb_id, cutoff=CONTACT_CUTOFF_A):
    if pdb_id not in filtered_acc.index:
        return None
    row       = filtered_acc.loc[pdb_id]
    pep_chain = row["peptide_chain"]
    rec_chains= set(c.strip() for c in str(row["receptor_chains"]).split(","))
    cif_path  = FIXED_DIR / f"{pdb_id}.cif"
    if not cif_path.exists():
        return None
    try:
        d      = MMCIF2Dict(str(cif_path))
        chains = d.get("_atom_site.auth_asym_id", [])
        xs     = list(map(float, d.get("_atom_site.Cartn_x", [])))
        ys     = list(map(float, d.get("_atom_site.Cartn_y", [])))
        zs     = list(map(float, d.get("_atom_site.Cartn_z", [])))
        pep = np.array([[x,y,z] for c,x,y,z in zip(chains,xs,ys,zs) if c == pep_chain],  dtype=np.float32)
        rec = np.array([[x,y,z] for c,x,y,z in zip(chains,xs,ys,zs) if c in rec_chains], dtype=np.float32)
        if not len(pep) or not len(rec):
            return 0
        dists, _ = cKDTree(rec).query(pep, k=1, workers=4)
        return int((dists < cutoff).sum())
    except Exception:
        return None

if CONTACTS_CACHE.exists():
    contacts_df = pd.read_csv(CONTACTS_CACHE, sep="\t")
    print(f"Loaded contacts cache: {len(contacts_df)} entries")
else:
    print("Computing contacts...")
    t0 = time.time()
    records = [{"pdb_id": pid, "n_contacts": compute_contacts(pid)}
               for pid in summary["pdb_id"]]
    contacts_df = pd.DataFrame(records)
    contacts_df.to_csv(CONTACTS_CACHE, sep="\t", index=False)
    print(f"Done in {time.time()-t0:.1f}s")

print(f"Missing CIF: {contacts_df['n_contacts'].isna().sum()}")

## 4. Feature matrix

In [ ]:
df = summary.merge(contacts_df, on="pdb_id", how="left")
df = df.dropna(subset=["n_contacts", TARGET]).copy()
df["n_contacts"]   = df["n_contacts"].astype(float)
df["resolution_f"] = df["resolution_f"].fillna(df["resolution_f"].median())

print(f"Working dataset : {len(df)} entries")
print(f"ΔG range        : {df[TARGET].min():.2f} to {df[TARGET].max():.2f} kcal/mol")
print(f"n_contacts      : {df['n_contacts'].min():.0f} to {df['n_contacts'].max():.0f}")
print(f"\nreceptor_func_class:")
print(df["receptor_func_class"].value_counts().to_string())

## 5. Sampling functions

In [ ]:
def _peptide_dedup(pool, threshold=PEP_DEDUP_THRESH):
    seqs, ids, kept, kept_seqs = pool["peptide_seq"].fillna("").tolist(), pool["pdb_id"].tolist(), [], []
    for pid, seq in zip(ids, seqs):
        if not any(SequenceMatcher(None, seq, ks).ratio() >= threshold for ks in kept_seqs):
            kept.append(pid); kept_seqs.append(seq)
    return pool[pool["pdb_id"].isin(kept)]


def _kennard_stone(X_sc, n):
    n    = min(n, len(X_sc))
    dist = cdist(X_sc, X_sc)
    i, j = np.unravel_index(dist.argmax(), dist.shape)
    sel  = [int(i), int(j)]
    rem  = list(set(range(len(X_sc))) - set(sel))
    while len(sel) < n:
        best = rem[int(dist[rem][:, sel].min(axis=1).argmax())]
        sel.append(best); rem.remove(best)
    return sel


def pepxpro_sample(pool, n):
    """Peptide sequence dedup + Kennard–Stone in 11D feature space."""
    deduped = _peptide_dedup(pool)
    if len(deduped) <= n:
        return deduped
    X    = deduped[FEATURES].values.astype(float)
    X_sc = StandardScaler().fit_transform(X)
    return deduped.iloc[_kennard_stone(X_sc, n)]


def internal_diversity(df_subset):
    """Mean pairwise Euclidean distance in standardized feature space."""
    X = StandardScaler().fit_transform(df_subset[FEATURES].values.astype(float))
    return cdist(X, X).mean()


df_deduped = _peptide_dedup(df)
print(f"After peptide dedup : {len(df_deduped)} entries ({len(df)-len(df_deduped)} removed)")

## 6. Main experiment

In [ ]:
results = []

print(f"Running {N_SEEDS} seeds  (80% train fixed, random vs KS test set n={N_TEST})...")
t0 = time.time()

for seed in range(N_SEEDS):
    np.random.seed(seed)
    idx        = np.random.permutation(len(df_deduped))
    train_size = int(TRAIN_FRAC * len(df_deduped))
    train_df   = df_deduped.iloc[idx[:train_size]]
    test_pool  = df_deduped.iloc[idx[train_size:]]

    if len(test_pool) < N_TEST:
        continue

    # ── Test set A: random ─────────────────────────────────────────
    test_A = test_pool.sample(n=N_TEST, random_state=seed)

    # ── Test set B: PepXPro (KS in 11D) ───────────────────────────
    test_B = pepxpro_sample(test_pool, N_TEST)

    # ── Train RF on fixed training set ────────────────────────────
    sc = StandardScaler()
    Xtr = sc.fit_transform(train_df[FEATURES].values.astype(float))
    rf  = RandomForestRegressor(n_estimators=200, min_samples_leaf=2,
                                random_state=42, n_jobs=-1)
    rf.fit(Xtr, train_df[TARGET].values)

    def rmse_on(test):
        yp = rf.predict(sc.transform(test[FEATURES].values.astype(float)))
        return float(np.sqrt(mean_squared_error(test[TARGET].values, yp)))

    results.append({
        "seed"       : seed,
        "rmse_rand"  : rmse_on(test_A),
        "rmse_ks"    : rmse_on(test_B),
        "div_rand"   : internal_diversity(test_A),
        "div_ks"     : internal_diversity(test_B),
        "dg_iqr_rand": float(test_A[TARGET].quantile(0.75) - test_A[TARGET].quantile(0.25)),
        "dg_iqr_ks"  : float(test_B[TARGET].quantile(0.75) - test_B[TARGET].quantile(0.25)),
        "dg_range_rand": float(test_A[TARGET].max() - test_A[TARGET].min()),
        "dg_range_ks"  : float(test_B[TARGET].max() - test_B[TARGET].min()),
    })

res = pd.DataFrame(results)
print(f"Done in {time.time()-t0:.1f}s   ({len(res)} seeds completed)")
print()
print(f"{'Metric':<30}  {'Random':>10}  {'PepXPro':>10}  {'Δ%':>7}")
print("-" * 62)
for col_a, col_b, label in [
    ("rmse_rand",   "rmse_ks",    "RMSE (↑ = harder)"),
    ("div_rand",    "div_ks",     "Internal diversity (↑ = more diverse)"),
    ("dg_iqr_rand", "dg_iqr_ks", "ΔG IQR  (↑ = wider)"),
    ("dg_range_rand","dg_range_ks","ΔG range (↑ = wider)"),
]:
    a, b = res[col_a].mean(), res[col_b].mean()
    pct  = (b - a) / abs(a) * 100
    print(f"{label:<30}  {a:>10.3f}  {b:>10.3f}  {pct:>+6.1f}%")
print()
pct_harder = (res["rmse_ks"] > res["rmse_rand"]).mean() * 100
print(f"PepXPro test set is harder in {pct_harder:.0f}% of seeds")

## 7. Sampling strategies — benchmark curation

**Design:**
- **Train**: stratified 80/20 split by ΔG (no redundancy removal) — RF trained on **all** ~591 entries
- **Test pool**: held-out 20% (~148 entries, no filters)
- **Test selection**: 6 strategies applied to the test pool, ranging from pure random to full PepXPro curation

Question: does a PepXPro-curated test set reveal harder / more diverse model evaluation than random?

In [ ]:
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

# ── Receptor sequence dedup (approx. CD-HIT 40%) ──────────────────────────
def _receptor_dedup(pool, threshold=0.40, window=150):
    seqs = pool["receptor_seq"].fillna("").tolist()
    ids  = pool["pdb_id"].tolist()
    kept, kept_seqs = [], []
    for pid, seq in zip(ids, seqs):
        sw, similar = seq[:window], False
        for ks in kept_seqs:
            if abs(len(seq) - len(ks)) / max(len(seq), len(ks), 1) > 0.6:
                continue
            if (SequenceMatcher(None, sw, ks[:window]).quick_ratio() >= threshold and
                    SequenceMatcher(None, sw, ks[:window]).ratio() >= threshold):
                similar = True; break
        if not similar:
            kept.append(pid); kept_seqs.append(seq)
    return pool[pool["pdb_id"].isin(kept)]

# ── Custom KS sampler (with optional pep-seq dedup) ───────────────────────
def ks_sample(pool, n, dims):
    d = _peptide_dedup(pool)
    if len(d) <= n: return d
    return d.iloc[_kennard_stone(StandardScaler().fit_transform(
        d[dims].values.astype(float)), n)]

# ── Stratified 80/20 split by dG — no redundancy removal ─────────────────
dg_bins = pd.qcut(df[TARGET], q=4, labels=False, duplicates="drop")
train_df, test_pool = train_test_split(
    df, test_size=0.20, stratify=dg_bins, random_state=42)

print(f"Train : {len(train_df)} entries  (no redundancy filter, all used)")
print(f"Test pool : {len(test_pool)} entries")
print(f"Train dG  : {train_df[TARGET].min():.1f} → {train_df[TARGET].max():.1f}")
print(f"Test  dG  : {test_pool[TARGET].min():.1f} → {test_pool[TARGET].max():.1f}")

# ── RF trained on all train entries ───────────────────────────────────────
sc_tr = StandardScaler()
Xtr   = sc_tr.fit_transform(train_df[FEATURES].values.astype(float))
rf    = RandomForestRegressor(n_estimators=200, min_samples_leaf=2,
                              random_state=42, n_jobs=-1)
rf.fit(Xtr, train_df[TARGET].values)

# ── Test pools after redundancy removal ───────────────────────────────────
test_seq     = _peptide_dedup(test_pool)           # pep-seq dedup only
test_seq_rec = _receptor_dedup(test_seq)           # + rec-seq dedup

N = 80
DIMS_1D = ["dG_kcal_mol"]
DIMS_2D = ["dG_kcal_mol", "pep_len"]
DIMS_3D = ["dG_kcal_mol", "pep_len", "n_contacts"]

strats = [
    ("Random",                 "#888888", test_pool.sample(n=N, random_state=42)),
    ("Seq + KS 1D\n[dG]",     "#9ecae1", ks_sample(test_pool, N, DIMS_1D)),
    ("Seq + KS 2D\n[dG,len]", "#4292c6", ks_sample(test_pool, N, DIMS_2D)),
    ("Seq + KS 3D\n[dG,len,ct]","#084594", ks_sample(test_pool, N, DIMS_3D)),
    ("Seq + KS 11D\n[all]",   "#c44e52", ks_sample(test_pool, N, FEATURES)),
    ("Seq+Rec\n+ KS 11D",     "#55a868", ks_sample(test_seq_rec, N, FEATURES)),
]

print(f"\nTest pool sizes after filtering:")
print(f"  raw : {len(test_pool)}  |  pep-seq dedup : {len(test_seq)}  |  + rec-seq dedup : {len(test_seq_rec)}")
print(f"\nTest set sizes per strategy:")
for label, _, ts in strats:
    print(f"  {label.replace(chr(10),' '):32s}  n={len(ts)}")

# ── PCA on test pool (background reference) ───────────────────────────────
sc_pca     = StandardScaler()
X_pool_sc  = sc_pca.fit_transform(test_pool[FEATURES].values.astype(float))
pca        = PCA(n_components=2, random_state=42)
X_pool_pca = pca.fit_transform(X_pool_sc)
var1, var2 = pca.explained_variance_ratio_[:2] * 100

# ── Figure: 2 rows × 6 cols ────────────────────────────────────────────────
NS = len(strats)
fig, axes = plt.subplots(2, NS, figsize=(3.2 * NS, 7.5))
fig.suptitle(
    f"Benchmark curation: test set selection strategies\n"
    f"Train = {len(train_df)} entries (stratified split, no redundancy filter) "
    f"· Fixed RF · Test pool = {len(test_pool)} entries",
    fontsize=10, fontweight="bold", y=1.01,
)

for col, (label, color, test) in enumerate(strats):
    y_true = test[TARGET].values
    y_pred = rf.predict(sc_tr.transform(test[FEATURES].values.astype(float)))
    rmse   = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r      = float(pearsonr(y_true, y_pred)[0])
    div    = internal_diversity(test)
    lim    = [test_pool[TARGET].min() - 0.3, test_pool[TARGET].max() + 0.3]

    # ── Row 0: pred vs exp ────────────────────────────────────────────────
    ax = axes[0, col]
    ax.scatter(y_true, y_pred, color=color, s=20, alpha=0.65, edgecolors="none")
    ax.plot(lim, lim, "k--", lw=0.7, alpha=0.35)
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
    ax.tick_params(labelsize=7)
    ax.set_title(f"{label}  (n={len(test)})\nRMSE={rmse:.2f}  r={r:.2f}", fontsize=8)
    if col == 0:
        ax.set_xlabel("Exp ΔG (kcal/mol)", fontsize=8)
        ax.set_ylabel("Pred ΔG (kcal/mol)", fontsize=8)

    # ── Row 1: PCA coverage ───────────────────────────────────────────────
    ax = axes[1, col]
    X_ts_pca = pca.transform(sc_pca.transform(test[FEATURES].values.astype(float)))
    ax.scatter(X_pool_pca[:, 0], X_pool_pca[:, 1],
               c="lightgray", s=10, alpha=0.4, zorder=1, label="Test pool")
    ax.scatter(X_ts_pca[:, 0], X_ts_pca[:, 1],
               c=color, s=28, alpha=0.85, zorder=2, label=f"Selected n={len(test)}")
    ax.tick_params(labelsize=7)
    ax.set_title(f"diversity={div:.2f}", fontsize=8)
    if col == 0:
        ax.set_xlabel(f"PC1 ({var1:.0f}%)", fontsize=8)
        ax.set_ylabel(f"PC2 ({var2:.0f}%)", fontsize=8)
    if col == NS - 1:
        ax.legend(fontsize=6, loc="upper right")

plt.tight_layout()
plt.savefig(PLOT_DIR / "07_sampling_strategies.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 07_sampling_strategies.png")


## 8. Save results

In [ ]:
test_pool.to_csv(OUT_DIR / "test_pool_raw.tsv", sep="	", index=False)

print("Output files:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  outputs/{f.name}")
for f in sorted(PLOT_DIR.iterdir()):
    print(f"  plots/{f.name}")
